# Lab 3: Neural Machine Translation — Transformer vs RNN

## Overview
This notebook implements two sequence-to-sequence (Seq2Seq) models for French→English neural machine translation:

1. **Transformer** — multi-head attention encoder-decoder built from scratch
2. **BiLSTM RNN** — bidirectional LSTM encoder with additive (Bahdanau) attention decoder

Both models are evaluated with **corpus-level BLEU** and include **attention visualization**.

## Data Setup
Before running, prepare the following directory structure:
```
data/
  train.fr   train.en
  validation.fr   validation.en   (or valid.fr / valid.en)
  test.fr    test.en
tokenizers/
  fr/    (HuggingFace BPE tokenizer for French)
  en/    (HuggingFace BPE tokenizer for English)
```

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers datasets matplotlib -q

In [ ]:
import os, math, time
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, PreTrainedTokenizerFast
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (8, 5)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Configuration

In [ ]:
@dataclass
class ConfigTransformer:
    hidden_size:          int   = 32
    intermediate_size:    int   = 128
    num_attention_heads:  int   = 4
    num_encoder_layers:   int   = 3
    num_decoder_layers:   int   = 3
    max_sequence_length:  int   = 32
    hidden_dropout_prob:  float = 0.1
    batch_size:           int   = 32
    learning_rate:        float = 1e-3
    max_epochs:           int   = 10

@dataclass
class ConfigRNN:
    embed_size:           int   = 256
    hidden_size:          int   = 512
    encoder_bidirectional: bool = True
    num_layers:           int   = 1
    lstm_dropout:         float = 0.3
    batch_size:           int   = 32
    learning_rate:        float = 1e-3
    max_epochs:           int   = 10

DATA_DIR       = './data'
TOKENIZER_FR   = './tokenizers/fr'
TOKENIZER_EN   = './tokenizers/en'
CHECKPOINT_DIR = './checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

cfgT = ConfigTransformer()
cfgR = ConfigRNN()

## 2. Tokenizers & Data Loading

In [ ]:
SPECIAL_TOKENS = {'pad_token': '<pad>', 'bos_token': '<bos>', 'eos_token': '<eos>'}

def load_tokenizer(local_dir: str) -> PreTrainedTokenizerFast:
    if not os.path.isdir(local_dir):
        raise FileNotFoundError(
            f"Tokenizer directory not found: {local_dir}\n"
            "Place the provided BPE tokenizer files there.")
    tok = AutoTokenizer.from_pretrained(local_dir, local_files_only=True)
    tok.add_special_tokens(SPECIAL_TOKENS)
    return tok

tok_fr = load_tokenizer(TOKENIZER_FR)
tok_en = load_tokenizer(TOKENIZER_EN)

PAD_ID_FR, PAD_ID_EN = tok_fr.pad_token_id, tok_en.pad_token_id
BOS_ID_EN, EOS_ID_EN = tok_en.bos_token_id, tok_en.eos_token_id
V_SIZE_FR, V_SIZE_EN = len(tok_fr), len(tok_en)
print(f"Vocab sizes — FR: {V_SIZE_FR}, EN: {V_SIZE_EN}")

In [ ]:
class ParallelTextDataset(Dataset):
    def __init__(self, fr_lines: List[str], en_lines: List[str],
                 max_src=cfgT.max_sequence_length, max_tgt=cfgT.max_sequence_length):
        assert len(fr_lines) == len(en_lines)
        self.fr, self.en = fr_lines, en_lines
        self.ms, self.mt = max_src, max_tgt

    def __len__(self): return len(self.fr)

    def __getitem__(self, i):
        src = torch.tensor(tok_fr.encode(self.fr[i], add_special_tokens=True)[:self.ms], dtype=torch.long)
        tgt = torch.tensor(tok_en.encode(self.en[i], add_special_tokens=True)[:self.mt], dtype=torch.long)
        return src, tgt

def read_lines(path): return [l.strip() for l in open(path, encoding='utf-8') if l.strip()]

def pad_batch(seqs, pad_id, max_len):
    out = [s[:max_len] + [pad_id]*(max_len - min(len(s), max_len)) for s in seqs]
    return torch.tensor(out, dtype=torch.long)

def collate(batch):
    src, tgt = zip(*batch)
    sp = pad_batch([x.tolist() for x in src], PAD_ID_FR, cfgT.max_sequence_length)
    tp = pad_batch([y.tolist() for y in tgt], PAD_ID_EN, cfgT.max_sequence_length)
    return sp, tp, (sp != PAD_ID_FR).long(), (tp != PAD_ID_EN).long()

# Load data files
def _load_split(name):
    # support both 'valid' and 'validation' filenames
    for variant in [name, name.replace('validation','valid')]:
        fp = os.path.join(DATA_DIR, f"{variant}.fr")
        ep = os.path.join(DATA_DIR, f"{variant}.en")
        if os.path.exists(fp) and os.path.exists(ep):
            return read_lines(fp), read_lines(ep)
    return [], []

fr_tr, en_tr = _load_split('train')
fr_va, en_va = _load_split('validation')
fr_te, en_te = _load_split('test')

N = min(len(fr_tr), len(en_tr))
train_ds     = ParallelTextDataset(fr_tr[:N], en_tr[:N])
train_loader = DataLoader(train_ds, batch_size=cfgT.batch_size, shuffle=True, collate_fn=collate)

valid_loader = test_pairs = None
if fr_va and en_va:
    valid_ds     = ParallelTextDataset(fr_va, en_va)
    valid_loader = DataLoader(valid_ds, batch_size=cfgT.batch_size, collate_fn=collate)
if fr_te and en_te:
    test_pairs = (fr_te, en_te)

print(f"Train: {N}, Val: {len(fr_va)}, Test: {len(fr_te)}")

## 3. BLEU Score

In [ ]:
def ngrams(tokens, n): return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def corpus_bleu(refs: List[List[str]], hyps: List[List[str]], max_n=4) -> float:
    p_ns = []
    for n in range(1, max_n+1):
        matches = total = 0
        for ref, hyp in zip(refs, hyps):
            rc, hc = Counter(ngrams(ref, n)), Counter(ngrams(hyp, n))
            matches += sum(min(c, rc.get(g, 0)) for g, c in hc.items())
            total   += sum(hc.values())
        p_ns.append(matches / max(total, 1))
    if any(p == 0 for p in p_ns): return 0.0
    geo = math.prod(p_ns) ** (1.0 / max_n)
    ref_len = sum(len(r) for r in refs)
    hyp_len = sum(len(h) for h in hyps)
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return bp * geo

## 4. Transformer Seq2Seq (from scratch)

In [ ]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d):
        super().__init__()
        self.pe = nn.Embedding(max_len, d)

    def forward(self, x):
        B, T, _ = x.size()
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        return x + self.pe(pos)


class MultiHeadAttentionScratch(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        assert d % n_heads == 0
        self.n_heads, self.hd = n_heads, d // n_heads
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.last_weights = None

    def forward(self, q, k, v, mask=None):
        B, Tq, H = q.size(); Tk = k.size(1)
        qh = self.q(q).view(B, Tq, self.n_heads, self.hd).transpose(1, 2)
        kh = self.k(k).view(B, Tk, self.n_heads, self.hd).transpose(1, 2)
        vh = self.v(v).view(B, Tk, self.n_heads, self.hd).transpose(1, 2)
        sc = torch.matmul(qh, kh.transpose(-2, -1)) / math.sqrt(self.hd)
        if mask is not None:
            if mask.dim() == 3: mask = mask.unsqueeze(1)
            sc = sc.masked_fill(mask == 0, float('-inf'))
        aw = torch.softmax(sc, dim=-1)
        self.last_weights = aw.detach()
        out = torch.matmul(aw, vh).transpose(1, 2).contiguous().view(B, Tq, H)
        return self.o(out)


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d, ff, h, drop):
        super().__init__()
        self.attn = MultiHeadAttentionScratch(d, h)
        self.ln1  = nn.LayerNorm(d)
        self.ff   = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Dropout(drop), nn.Linear(ff, d))
        self.ln2  = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)

    def forward(self, x, src_mask):
        B, T, _ = x.size()
        m = src_mask.unsqueeze(1).unsqueeze(2).expand(B, 1, T, T)
        x = self.ln1(x + self.drop(self.attn(x, x, x, m)))
        return self.ln2(x + self.drop(self.ff(x)))


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d, ff, h, drop):
        super().__init__()
        self.self_attn  = MultiHeadAttentionScratch(d, h)
        self.cross_attn = MultiHeadAttentionScratch(d, h)
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d); self.ln3 = nn.LayerNorm(d)
        self.ff  = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Dropout(drop), nn.Linear(ff, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, mem, tgt_mask, mem_mask):
        B, T, _  = x.size(); S = mem.size(1)
        causal   = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        self_m   = causal * tgt_mask.unsqueeze(1).unsqueeze(2)
        x = self.ln1(x + self.drop(self.self_attn(x, x, x, self_m)))
        cross_m  = mem_mask.unsqueeze(1).unsqueeze(2).expand(B, 1, T, S)
        x = self.ln2(x + self.drop(self.cross_attn(x, mem, mem, cross_m)))
        return self.ln3(x + self.drop(self.ff(x)))


class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, cfg: ConfigTransformer):
        super().__init__()
        H = cfg.hidden_size
        self.src_emb = nn.Embedding(src_vocab, H)
        self.tgt_emb = nn.Embedding(tgt_vocab, H)
        self.pos_enc = LearnedPositionalEmbedding(cfg.max_sequence_length, H)
        self.pos_dec = LearnedPositionalEmbedding(cfg.max_sequence_length, H)
        args = (H, cfg.intermediate_size, cfg.num_attention_heads, cfg.hidden_dropout_prob)
        self.enc_layers  = nn.ModuleList([TransformerEncoderLayer(*args) for _ in range(cfg.num_encoder_layers)])
        self.dec_layers  = nn.ModuleList([TransformerDecoderLayer(*args) for _ in range(cfg.num_decoder_layers)])
        self.ln_enc = nn.LayerNorm(H); self.ln_dec = nn.LayerNorm(H)
        self.lm_head = nn.Linear(H, tgt_vocab, bias=False)
        self.lm_head.weight = self.tgt_emb.weight     # weight tying
        self.last_cross_attn: List = []

    def encode(self, src, src_mask):
        x = self.pos_enc(self.src_emb(src))
        for l in self.enc_layers: x = l(x, src_mask)
        return self.ln_enc(x)

    def decode(self, tgt, mem, tgt_mask, src_mask):
        x = self.pos_dec(self.tgt_emb(tgt))
        self.last_cross_attn = []
        for l in self.dec_layers:
            x = l(x, mem, tgt_mask, src_mask)
            self.last_cross_attn.append(l.cross_attn.last_weights)
        return self.lm_head(self.ln_dec(x))

    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(tgt, self.encode(src, src_mask), tgt_mask, src_mask)

    @torch.no_grad()
    def generate(self, src, src_mask, bos=BOS_ID_EN, eos=EOS_ID_EN, max_len=32, beam=4):
        B = src.size(0); mem = self.encode(src, src_mask)
        beams    = [[(0.0, [bos])] for _ in range(B)]
        finished = [[] for _ in range(B)]
        for _ in range(max_len):
            new_beams = [[] for _ in range(B)]
            for i in range(B):
                cands = []
                for score, seq in beams[i]:
                    if seq[-1] == eos: finished[i].append((score, seq)); continue
                    t  = torch.tensor(seq, device=mem.device).unsqueeze(0)
                    tm = (t != PAD_ID_EN).long()
                    lp = F.log_softmax(self.decode(t, mem[i:i+1], tm, src_mask[i:i+1])[:, -1], -1)
                    topk = torch.topk(lp, beam, -1)
                    cands += [(score + topk.values[0, k].item(), seq + [int(topk.indices[0, k])]) for k in range(beam)]
                cands.sort(key=lambda x: x[0], reverse=True)
                new_beams[i] = cands[:beam]
            beams = new_beams
        out = []
        for i in range(B):
            all_c = finished[i] + beams[i]
            all_c = [(s/len(sq), sq) for s, sq in all_c]
            out.append(max(all_c, key=lambda x: x[0])[1])
        return out

## 5. RNN Seq2Seq (BiLSTM + Additive Attention)

In [ ]:
class EncoderBiLSTM(nn.Module):
    def __init__(self, vocab, embed, hidden, layers=1, bidir=True, drop=0.0):
        super().__init__()
        self.emb  = nn.Embedding(vocab, embed)
        self.lstm = nn.LSTM(embed, hidden, num_layers=layers, bidirectional=bidir,
                            batch_first=True, dropout=drop if layers > 1 else 0.0)
        self.bidir, self.layers, self.hidden = bidir, layers, hidden

    def forward(self, src, src_mask):
        return self.lstm(self.emb(src))

class AdditiveAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.W_enc = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_dec = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v     = nn.Linear(attn_dim, 1, bias=False)
        self.last_weights = None

    def forward(self, enc_out, dec_h, mask):
        B, T, _ = enc_out.size()
        sc = self.v(torch.tanh(self.W_enc(enc_out) + self.W_dec(dec_h).unsqueeze(1).expand(B, T, -1))).squeeze(-1)
        sc = sc.masked_fill(mask == 0, float('-inf'))
        w  = torch.softmax(sc, -1); self.last_weights = w.detach()
        return torch.bmm(w.unsqueeze(1), enc_out).squeeze(1), w

class DecoderLSTM(nn.Module):
    def __init__(self, vocab, embed, hidden, enc_out, attn_dim, layers=1, drop=0.0):
        super().__init__()
        self.emb  = nn.Embedding(vocab, embed)
        self.lstm = nn.LSTM(embed + enc_out, hidden, num_layers=layers, batch_first=True,
                            dropout=drop if layers > 1 else 0.0)
        self.attn = AdditiveAttention(enc_out, hidden, attn_dim)
        self.proj = nn.Linear(hidden + enc_out + embed, vocab)
        self.last_attn_weights = None

    def forward(self, tgt, enc_out, enc_mask, state):
        x = self.emb(tgt); h, c = state; outs = []; ws = []
        for t in range(tgt.size(1)):
            xt = x[:, t:t+1, :]
            ctx, w = self.attn(enc_out, h[-1], enc_mask); ws.append(w)
            out, (h, c) = self.lstm(torch.cat([xt, ctx.unsqueeze(1)], -1), (h, c))
            outs.append(self.proj(torch.cat([out[:, -1], ctx, xt.squeeze(1)], -1)).unsqueeze(1))
        self.last_attn_weights = torch.stack(ws, 1)
        return torch.cat(outs, 1)

class Seq2SeqRNN(nn.Module):
    def __init__(self, src_v, tgt_v, cfg: ConfigRNN):
        super().__init__()
        self.enc     = EncoderBiLSTM(src_v, cfg.embed_size, cfg.hidden_size,
                                     cfg.num_layers, cfg.encoder_bidirectional, cfg.lstm_dropout)
        enc_out_dim  = cfg.hidden_size * (2 if cfg.encoder_bidirectional else 1)
        self.enc_dim = enc_out_dim
        self.proj_h  = nn.Linear(enc_out_dim, cfg.hidden_size)
        self.proj_c  = nn.Linear(enc_out_dim, cfg.hidden_size)
        self.dec     = DecoderLSTM(tgt_v, cfg.embed_size, cfg.hidden_size, enc_out_dim, cfg.hidden_size)

    def encode(self, src, src_mask):
        enc_out, (h, c) = self.enc(src, src_mask)
        if self.enc.bidir:
            h_l = torch.cat([h[-2], h[-1]], -1); c_l = torch.cat([c[-2], c[-1]], -1)
        else:
            h_l, c_l = h[-1], c[-1]
        return enc_out, (torch.tanh(self.proj_h(h_l)).unsqueeze(0),
                         torch.tanh(self.proj_c(c_l)).unsqueeze(0))

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_out, st = self.encode(src, src_mask)
        return self.dec(tgt, enc_out, src_mask, st)

    @torch.no_grad()
    def generate(self, src, src_mask, bos=BOS_ID_EN, eos=EOS_ID_EN, max_len=32, beam=4):
        B = src.size(0); enc_out, (h0, c0) = self.encode(src, src_mask)
        beams    = [[(0.0, [bos], (h0[:,i:i+1].contiguous(), c0[:,i:i+1].contiguous()))] for i in range(B)]
        finished = [[] for _ in range(B)]
        for _ in range(max_len):
            new_beams = [[] for _ in range(B)]
            for i in range(B):
                cands = []
                for score, seq, (h, c) in beams[i]:
                    if seq[-1] == eos: finished[i].append((score, seq)); continue
                    xt  = torch.tensor([seq[-1]], device=enc_out.device).unsqueeze(0)
                    emb = self.dec.emb(xt)
                    ctx, _ = self.dec.attn(enc_out[i:i+1], h[-1], src_mask[i:i+1])
                    out, (hn, cn) = self.dec.lstm(torch.cat([emb, ctx.unsqueeze(1)], -1), (h, c))
                    logit = self.dec.proj(torch.cat([out[:,-1], ctx, emb.squeeze(1)], -1))
                    lp    = F.log_softmax(logit, -1)
                    topk  = torch.topk(lp, beam, -1)
                    cands += [(score + topk.values[0,k].item(), seq+[int(topk.indices[0,k])], (hn,cn)) for k in range(beam)]
                cands.sort(key=lambda x: x[0], reverse=True)
                new_beams[i] = cands[:beam]
            beams = new_beams
        out = []
        for i in range(B):
            all_c = finished[i] + [(s, sq) for s, sq, _ in beams[i]]
            all_c = [(s/len(sq), sq) for s, sq in all_c]
            out.append(max(all_c, key=lambda x: x[0])[1])
        return out

## 6. Training Utilities

In [ ]:
def shifted_cross_entropy(logits, tgt, pad_id):
    lg = logits[:, :-1, :]; tg = tgt[:, 1:]
    B, T, V = lg.size()
    return F.cross_entropy(lg.reshape(B*T, V), tg.reshape(-1), ignore_index=pad_id)

def save_ckpt(path, model, opt, extra):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'model': model.state_dict(),
                'opt':   opt.state_dict() if opt else None, 'extra': extra}, path)
    print(f"  Saved → {path}")

def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ck['model'])
    if opt and ck.get('opt'): opt.load_state_dict(ck['opt'])
    return ck.get('extra', {})

def train_one_epoch(model, loader, opt, pad_id):
    model.train(); total = 0.0
    for src, tgt, sm, tm in loader:
        src, tgt, sm, tm = src.to(DEVICE), tgt.to(DEVICE), sm.to(DEVICE), tm.to(DEVICE)
        opt.zero_grad()
        loss = shifted_cross_entropy(model(src, tgt, sm, tm), tgt, pad_id)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item()
    return total / len(loader)

@torch.no_grad()
def bleu_on_texts(model, fr_lines, en_lines, max_len=32):
    model.eval()
    batch = tok_fr.batch_encode_plus(fr_lines, add_special_tokens=True)['input_ids']
    src   = pad_batch(batch, PAD_ID_FR, max_len).to(DEVICE)
    sm    = (src != PAD_ID_FR).long()
    hyp_ids = model.generate(src, sm, max_len=max_len)
    hyps  = [tok_en.decode(ids) for ids in hyp_ids]
    refs  = [r.split() for r in en_lines]
    hyps_tok = [h.split() for h in hyps]
    return corpus_bleu(refs, hyps_tok), hyps

## 7. Instantiate & Train Both Models

In [ ]:
transformer = TransformerSeq2Seq(V_SIZE_FR, V_SIZE_EN, cfgT).to(DEVICE)
rnn         = Seq2SeqRNN(V_SIZE_FR, V_SIZE_EN, cfgR).to(DEVICE)

opt_T = torch.optim.Adam(transformer.parameters(), lr=cfgT.learning_rate)
opt_R = torch.optim.Adam(rnn.parameters(),         lr=cfgR.learning_rate)

T_losses, R_losses = [], []

for epoch in range(1, cfgT.max_epochs + 1):
    t0 = time.time()
    lT = train_one_epoch(transformer, train_loader, opt_T, PAD_ID_EN)
    lR = train_one_epoch(rnn,         train_loader, opt_R, PAD_ID_EN)
    T_losses.append(lT); R_losses.append(lR)
    save_ckpt(f"{CHECKPOINT_DIR}/transformer_ep{epoch}.pt", transformer, opt_T, {'epoch': epoch})
    save_ckpt(f"{CHECKPOINT_DIR}/rnn_ep{epoch}.pt",         rnn,         opt_R, {'epoch': epoch})
    print(f"Epoch {epoch}/{cfgT.max_epochs}  Transformer={lT:.4f}  RNN={lR:.4f}  ({time.time()-t0:.1f}s)")

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(T_losses, label='Transformer'); plt.plot(R_losses, label='RNN')
plt.title('Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('nmt_training_loss.png', dpi=150); plt.show()

## 8. BLEU Evaluation

In [ ]:
print("Evaluating BLEU …")
bleu_T_tr, _ = bleu_on_texts(transformer, fr_tr[:500], en_tr[:500])
bleu_R_tr, _ = bleu_on_texts(rnn,         fr_tr[:500], en_tr[:500])
print(f"Train BLEU — Transformer: {bleu_T_tr:.4f} | RNN: {bleu_R_tr:.4f}")

if fr_va:
    bleu_T_va, _ = bleu_on_texts(transformer, fr_va, en_va)
    bleu_R_va, _ = bleu_on_texts(rnn,         fr_va, en_va)
    print(f"Valid BLEU — Transformer: {bleu_T_va:.4f} | RNN: {bleu_R_va:.4f}")

if test_pairs:
    bleu_T_te, _ = bleu_on_texts(transformer, *test_pairs)
    bleu_R_te, _ = bleu_on_texts(rnn,         *test_pairs)
    print(f"Test  BLEU — Transformer: {bleu_T_te:.4f} | RNN: {bleu_R_te:.4f}")

## 9. Attention Visualization

In [ ]:
def show_transformer_attention(model, src_text, tgt_text=None, layer=0, head=0):
    model.eval()
    src = pad_batch([tok_fr.encode(src_text, add_special_tokens=True)], PAD_ID_FR, cfgT.max_sequence_length).to(DEVICE)
    sm  = (src != PAD_ID_FR).long()
    if tgt_text:
        tgt = pad_batch([tok_en.encode(tgt_text, add_special_tokens=True)], PAD_ID_EN, cfgT.max_sequence_length).to(DEVICE)
    else:
        tgt = pad_batch([[BOS_ID_EN]], PAD_ID_EN, cfgT.max_sequence_length).to(DEVICE)
    tm  = (tgt != PAD_ID_EN).long()
    model.decode(tgt, model.encode(src, sm), tm, sm)
    attn = model.last_cross_attn[layer][0, head].cpu().numpy()
    src_tok = tok_fr.decode(src[0].tolist()).split()
    tgt_tok = tok_en.decode(tgt[0].tolist()).split()
    plt.figure(figsize=(8, 6))
    plt.imshow(attn[:len(tgt_tok), :len(src_tok)], aspect='auto', cmap='viridis')
    plt.colorbar(); plt.xticks(range(len(src_tok)), src_tok, rotation=45)
    plt.yticks(range(len(tgt_tok)), tgt_tok)
    plt.title(f'Transformer Cross-Attention — Layer {layer} Head {head}')
    plt.xlabel('Source (FR)'); plt.ylabel('Target (EN)'); plt.tight_layout(); plt.show()

def show_rnn_attention(model, src_text, tgt_text):
    model.eval()
    src = pad_batch([tok_fr.encode(src_text, add_special_tokens=True)], PAD_ID_FR, cfgT.max_sequence_length).to(DEVICE)
    tgt = pad_batch([tok_en.encode(tgt_text, add_special_tokens=True)], PAD_ID_EN, cfgT.max_sequence_length).to(DEVICE)
    sm, tm = (src != PAD_ID_FR).long(), (tgt != PAD_ID_EN).long()
    with torch.no_grad(): model(src, tgt, sm, tm)
    attn = model.dec.last_attn_weights[0].cpu().numpy()
    src_tok = tok_fr.decode(src[0].tolist()).split()
    tgt_tok = tok_en.decode(tgt[0].tolist()).split()
    plt.figure(figsize=(8, 6))
    plt.imshow(attn[:len(tgt_tok), :len(src_tok)], aspect='auto', cmap='magma')
    plt.colorbar(); plt.xticks(range(len(src_tok)), src_tok, rotation=45)
    plt.yticks(range(len(tgt_tok)), tgt_tok)
    plt.title('RNN Additive Attention')
    plt.xlabel('Source (FR)'); plt.ylabel('Target (EN)'); plt.tight_layout(); plt.show()

# Example — adjust sentences to ones in your dataset
examples = ['bonjour', 'je suis etudiant', 'mon nom est chat']
print("Transformer translations:")
_, hyps_T = bleu_on_texts(transformer, examples, ['']*len(examples))
for fr, en in zip(examples, hyps_T): print(f"  {fr!r:30s} → {en!r}")

print("RNN translations:")
_, hyps_R = bleu_on_texts(rnn, examples, ['']*len(examples))
for fr, en in zip(examples, hyps_R): print(f"  {fr!r:30s} → {en!r}")